# NexaMem Quick Start with Azure Redis - Enterprise Authentication

This notebook demonstrates how to use **NexaMem** (Next-Generation AI Memory) with **Azure Cache for Redis** using **Entra ID (Azure AD) authentication** for enterprise-grade security and production deployments.

## What is NexaMem?

NexaMem is a Python package for managing AI chatbot conversation history with:
- Multiple storage backends (memory, file, SQLite, Redis)
- Channel-based memory organization  
- Advanced features like encryption, quotas, and TTL
- Enterprise Azure Cloud integration

## Prerequisites

Before running this notebook, ensure you have:
1. An **Azure Cache for Redis** instance with **Entra ID enabled**
2. **Azure identity** configured (Managed Identity, Service Principal, or User Identity)
3. **Proper Azure permissions** for Redis access
4. **Azure CLI or authentication** setup

## When to Use This Approach

**Entra ID authentication** is ideal for:
- 🏢 **Production environments** with enterprise security requirements
- 🔐 **Zero-trust architectures** with identity-based access
- 🎯 **Managed Identity** scenarios (Azure VMs, App Service, Functions)
- 🔑 **Service Principal** automation and CI/CD pipelines
- 🛡️ **Compliance** requirements (SOC2, HIPAA, etc.)
- 🔄 **Token rotation** and centralized identity management

For simpler development and testing, consider the [Basic notebook](quickstart_azure_redis_basic.ipynb) with access key authentication.

## 1. Install Required Dependencies

For enterprise features, we'll install external dependencies and use the **local development version** of NexaMem from this repository. This ensures you have the latest enterprise features and security fixes.

**What we'll install:**
- `redis` - Python Redis client
- `azure-identity` - Enterprise Azure authentication (Entra ID, Managed Identity, Service Principal)
- `azure-keyvault-secrets` - Azure Key Vault integration for secure secret management

**What we'll use locally:**
- `nexamem` - From the `src/nexamem/` directory in this repository (latest enterprise features)

In [1]:
# Install dependencies and use local development version of NexaMem
# Only install external dependencies from pip, use local NexaMem from repo
!pip install redis

# Enterprise: Azure identity packages for Entra ID authentication
!pip install azure-identity azure-keyvault-secrets

print("✅ Enterprise dependencies installed!")
print("📦 Installed: redis, azure-identity, azure-keyvault-secrets")
print("ℹ️  Using local development version of NexaMem from repository")
print("   This ensures you have the latest enterprise features and fixes")

✅ Enterprise dependencies installed!
📦 Installed: redis, azure-identity, azure-keyvault-secrets
ℹ️  Using local development version of NexaMem from repository
   This ensures you have the latest enterprise features and fixes
✅ Enterprise dependencies installed!
📦 Installed: redis, azure-identity, azure-keyvault-secrets
ℹ️  Using local development version of NexaMem from repository
   This ensures you have the latest enterprise features and fixes


## 2. Import Required Modules

In [2]:
import os
import sys
import json
import tempfile
from typing import Dict, Any

# Add local src directory to path for development version
repo_root = os.getcwd()
src_path = os.path.join(repo_root, 'src')

print(f"📁 Repository root: {repo_root}")
print(f"📁 Source path: {src_path}")
print(f"📁 Source exists: {os.path.exists(src_path)}")

if os.path.exists(src_path):
    # Insert at beginning of path to prioritize local version
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
        print(f"✅ Added {src_path} to Python path")
    
    # Force reload of modules if they're already loaded to ensure we get the latest local version
    modules_to_reload = ['nexamem', 'nexamem.redis_adapter', 'nexamem.history', 'nexamem.aimemory', 'nexamem.config']
    for module in modules_to_reload:
        if module in sys.modules:
            del sys.modules[module]
            print(f"🔄 Reloaded module: {module}")
else:
    print("⚠️  Local src directory not found - will try to use installed package")

# NexaMem imports from local development version
try:
    from nexamem import initialize, ChatHistory, AIMemory, MemoryScope
    from nexamem.redis_adapter import AzureRedisConnectionHelper
    
    print("✅ NexaMem imports successful!")
    
    # Show which version we're actually using
    import nexamem
    if hasattr(nexamem, '__file__'):
        nexamem_path = nexamem.__file__
        print(f"📍 Using NexaMem from: {nexamem_path}")
        
        # Check if it's the local development version
        if src_path.lower() in nexamem_path.lower():
            print("🚀 Confirmed: Using LOCAL DEVELOPMENT version")
        else:
            print("⚠️  Warning: Using installed package version")
    
    # Show Redis adapter source
    import nexamem.redis_adapter
    if hasattr(nexamem.redis_adapter, '__file__'):
        print(f"📍 Redis adapter from: {nexamem.redis_adapter.__file__}")
        
except ImportError as e:
    print(f"❌ NexaMem import failed: {e}")
    print("   Make sure the src/nexamem directory exists and contains the modules")

# Azure identity imports
try:
    from azure.identity import DefaultAzureCredential, ClientSecretCredential, ManagedIdentityCredential
    from azure.keyvault.secrets import SecretClient
    azure_identity_available = True
    print("✅ Azure identity packages loaded successfully")
except ImportError as e:
    azure_identity_available = False
    print(f"❌ Azure identity packages not available: {e}")
    print("   Run: pip install azure-identity azure-keyvault-secrets")

print("\n🏢 Enterprise setup complete!")
print(f"   🚀 Local NexaMem: {'✅' if 'nexamem' in sys.modules else '❌'}")
print(f"   🔐 Azure Identity: {'✅' if azure_identity_available else '❌'}")
print(f"   🛡️  Enterprise Ready: {'✅' if azure_identity_available and 'nexamem' in sys.modules else '❌'}")

📁 Repository root: c:\Users\chadada\source\repos\nexamem
📁 Source path: c:\Users\chadada\source\repos\nexamem\src
📁 Source exists: True
✅ Added c:\Users\chadada\source\repos\nexamem\src to Python path
✅ NexaMem imports successful!
📍 Using NexaMem from: c:\Users\chadada\source\repos\nexamem\src\nexamem\__init__.py
🚀 Confirmed: Using LOCAL DEVELOPMENT version
📍 Redis adapter from: c:\Users\chadada\source\repos\nexamem\src\nexamem\redis_adapter.py
✅ Azure identity packages loaded successfully

🏢 Enterprise setup complete!
   🚀 Local NexaMem: ✅
   🔐 Azure Identity: ✅
   🛡️  Enterprise Ready: ✅
✅ NexaMem imports successful!
📍 Using NexaMem from: c:\Users\chadada\source\repos\nexamem\src\nexamem\__init__.py
🚀 Confirmed: Using LOCAL DEVELOPMENT version
📍 Redis adapter from: c:\Users\chadada\source\repos\nexamem\src\nexamem\redis_adapter.py
✅ Azure identity packages loaded successfully

🏢 Enterprise setup complete!
   🚀 Local NexaMem: ✅
   🔐 Azure Identity: ✅
   🛡️  Enterprise Ready: ✅


## 3. Azure Identity Configuration

### 🔐 Entra ID Authentication Options

Choose the authentication method that best fits your deployment:

### Option A: Managed Identity (Recommended for Azure Services)

In [ ]:
# Managed Identity Configuration (Azure VMs, App Service, Functions)
# This is the most secure option for Azure-hosted applications

MANAGED_IDENTITY_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        # 🔄 Replace with your Azure Redis hostname
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        
        # Enable Entra ID authentication
        "use_azure_entra_id": True,
        
        # For system-assigned managed identity (most common)
        "azure_username": "default",
        
        # For user-assigned managed identity (uncomment if needed)
        # "azure_client_id": "your-user-assigned-managed-identity-client-id",
        
        "key_prefix": "nexamem:managed-identity",
        "db": 0
    },
    "debug": True
}

print("🏢 Managed Identity configuration prepared")
print("   Best for: Azure VMs, App Service, Functions, Container Instances")
print("   Security: Highest - No secrets to manage")
print(f"   Hostname: {MANAGED_IDENTITY_CONFIG['redis_config']['azure_hostname']}")

### Option B: Service Principal (For Automation & CI/CD)

In [ ]:
# Service Principal Configuration (for automation, CI/CD, external apps)
# Requires App Registration in Azure AD

SERVICE_PRINCIPAL_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        # 🔄 Replace with your Azure Redis hostname
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        
        # Enable Entra ID authentication
        "use_azure_entra_id": True,
        
        # Service Principal credentials
        # 🚨 Replace with your App Registration details
        "azure_client_id": "YOUR-CLIENT-ID-HERE",
        "azure_client_secret": "YOUR-CLIENT-SECRET-HERE",
        "azure_tenant_id": "YOUR-TENANT-ID-HERE",
        "azure_username": "YOUR-CLIENT-ID-HERE",  # Usually same as client_id
        
        "key_prefix": "nexamem:service-principal",
        "db": 0
    },
    "debug": True
}

print("🤖 Service Principal configuration prepared")
print("   Best for: CI/CD pipelines, external applications, automation")
print("   Security: High - Requires secret management")
print(f"   Hostname: {SERVICE_PRINCIPAL_CONFIG['redis_config']['azure_hostname']}")
print("\n📋 Required App Registration permissions:")
print("   - Azure Cache for Redis: Data Contributor or Data Reader")

### Option C: Azure Key Vault Integration (Enterprise Secret Management)

In [ ]:
# Azure Key Vault Configuration (for centralized secret management)
# Stores Redis access key in Key Vault, retrieved using Entra ID

KEY_VAULT_CONFIG = {
    "history_storage": "redis",
    "redis_config": {
        # 🔄 Replace with your Azure Redis hostname
        "azure_hostname": "YOUR-REDIS-NAME.redis.cache.windows.net",
        
        # Azure Key Vault configuration
        # 🔑 Replace with your Key Vault details
        "azure_keyvault_url": "https://YOUR-KEYVAULT-NAME.vault.azure.net/",
        "azure_secret_name": "redis-access-key",
        
        "key_prefix": "nexamem:keyvault",
        "db": 0
    },
    "debug": True
}

print("🔐 Key Vault configuration prepared")
print("   Best for: Centralized secret management, compliance requirements")
print("   Security: Very High - Secrets stored in Key Vault")
print(f"   Hostname: {KEY_VAULT_CONFIG['redis_config']['azure_hostname']}")
print(f"   Key Vault: {KEY_VAULT_CONFIG['redis_config']['azure_keyvault_url']}")
print("\n📋 Required permissions:")
print("   - Key Vault: Get Secrets permission")
print("   - Redis: Access via stored key")

### Option D: Environment Variables (Production Deployment)

In [ ]:
# Environment-based configuration for production deployments
# Supports all authentication methods via environment variables

def create_enterprise_config_from_env():
    """Create enterprise Redis configuration from environment variables."""
    hostname = os.getenv('AZURE_REDIS_HOSTNAME')
    
    if not hostname:
        print("❌ Missing required environment variable: AZURE_REDIS_HOSTNAME")
        return None
    
    config = {
        "history_storage": "redis",
        "redis_config": {
            "azure_hostname": hostname,
            "key_prefix": "nexamem:production",
            "db": 0
        },
        "debug": False
    }
    
    # Check for Key Vault configuration
    keyvault_url = os.getenv('AZURE_KEYVAULT_URL')
    secret_name = os.getenv('AZURE_REDIS_SECRET_NAME')
    
    if keyvault_url and secret_name:
        config['redis_config']['azure_keyvault_url'] = keyvault_url
        config['redis_config']['azure_secret_name'] = secret_name
        print(f"✅ Key Vault configuration: {keyvault_url}")
        return config
    
    # Check for Service Principal configuration
    client_id = os.getenv('AZURE_CLIENT_ID')
    client_secret = os.getenv('AZURE_CLIENT_SECRET')
    tenant_id = os.getenv('AZURE_TENANT_ID')
    
    if client_id and client_secret and tenant_id:
        config['redis_config'].update({
            "use_azure_entra_id": True,
            "azure_client_id": client_id,
            "azure_client_secret": client_secret,
            "azure_tenant_id": tenant_id,
            "azure_username": client_id
        })
        print(f"✅ Service Principal configuration: {client_id}")
        return config
    
    # Default to Managed Identity
    config['redis_config'].update({
        "use_azure_entra_id": True,
        "azure_username": "default"
    })
    print("✅ Managed Identity configuration (default)")
    return config

ENV_ENTERPRISE_CONFIG = create_enterprise_config_from_env()

if ENV_ENTERPRISE_CONFIG:
    print("\n🌍 Environment-based enterprise configuration ready")
else:
    print("\nℹ️  Environment variables not set - use manual configuration")
    print("\n📝 Required environment variables:")
    print("   AZURE_REDIS_HOSTNAME=your-redis.redis.cache.windows.net")
    print("\n🔐 Authentication options (choose one):")
    print("   Managed Identity: (no additional vars needed)")
    print("   Service Principal: AZURE_CLIENT_ID, AZURE_CLIENT_SECRET, AZURE_TENANT_ID")
    print("   Key Vault: AZURE_KEYVAULT_URL, AZURE_REDIS_SECRET_NAME")

## 4. Test Authentication Configuration

Let's validate the Entra ID authentication setup:

In [ ]:
# Choose configuration to test (modify as needed)
# Options: MANAGED_IDENTITY_CONFIG, SERVICE_PRINCIPAL_CONFIG, KEY_VAULT_CONFIG, ENV_ENTERPRISE_CONFIG
test_config = ENV_ENTERPRISE_CONFIG if ENV_ENTERPRISE_CONFIG else MANAGED_IDENTITY_CONFIG

print(f"🧪 Testing configuration...")
print(f"   Auth method: {test_config['redis_config'].get('use_azure_entra_id', 'Key Vault' if 'azure_keyvault_url' in test_config['redis_config'] else 'Unknown')}")

try:
    if not azure_identity_available:
        raise ImportError("Azure identity packages not available")
    
    # Test Azure credential acquisition
    if 'azure_keyvault_url' in test_config['redis_config']:
        # Key Vault approach
        print("🔐 Testing Key Vault access...")
        keyvault_url = test_config['redis_config']['azure_keyvault_url']
        secret_name = test_config['redis_config']['azure_secret_name']
        
        credential = DefaultAzureCredential()
        # Note: This will fail if not properly authenticated, which is expected
        print(f"   Key Vault URL: {keyvault_url}")
        print(f"   Secret name: {secret_name}")
        print("   ✅ Key Vault configuration valid")
        
    elif test_config['redis_config'].get('use_azure_entra_id'):
        # Entra ID approach
        if 'azure_client_secret' in test_config['redis_config']:
            print("🤖 Testing Service Principal authentication...")
            client_id = test_config['redis_config']['azure_client_id']
            tenant_id = test_config['redis_config']['azure_tenant_id']
            print(f"   Client ID: {client_id}")
            print(f"   Tenant ID: {tenant_id}")
            print("   ✅ Service Principal configuration valid")
        else:
            print("🏢 Testing Managed Identity authentication...")
            username = test_config['redis_config']['azure_username']
            print(f"   Username: {username}")
            print("   ✅ Managed Identity configuration valid")
    
    # Test Redis configuration generation
    hostname = test_config['redis_config']['azure_hostname']
    
    if hostname.startswith('YOUR-'):
        print("\n⚠️  Placeholder hostname detected - replace with actual Redis hostname")
    else:
        print("\n🔗 Redis connection configuration:")
        print(f"   Hostname: {hostname}")
        print(f"   Port: 6380 (SSL)")
        print(f"   Authentication: Entra ID")
        print(f"   Key prefix: {test_config['redis_config']['key_prefix']}")
        
except Exception as e:
    print(f"❌ Configuration test failed: {e}")
    print("\n🔧 Troubleshooting:")
    print("   1. Ensure Azure CLI is logged in: az login")
    print("   2. Check Azure permissions for Redis and Key Vault")
    print("   3. Verify Managed Identity is enabled (if using Azure services)")
    print("   4. Confirm Service Principal credentials are correct")

## 5. Initialize NexaMem with Enterprise Authentication

Now let's initialize NexaMem with Entra ID authentication:

In [ ]:
# Choose your configuration (modify as needed)
config_to_use = ENV_ENTERPRISE_CONFIG if ENV_ENTERPRISE_CONFIG else MANAGED_IDENTITY_CONFIG

try:
    # Initialize NexaMem with enterprise authentication
    initialize(config_to_use)
    print("🎉 NexaMem successfully initialized with Entra ID authentication!")
    
    # Test the connection
    history = ChatHistory()
    print("✅ ChatHistory instance created successfully")
    
    # Display authentication details
    auth_method = "Key Vault" if 'azure_keyvault_url' in config_to_use['redis_config'] else "Entra ID"
    print(f"\n🔐 Authentication: {auth_method}")
    print(f"🏷️  Key prefix: {config_to_use['redis_config']['key_prefix']}")
    print(f"🌍 Environment: Enterprise/Production-ready")
    
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    print("\n🔧 Enterprise troubleshooting checklist:")
    print("   1. ✓ Azure Redis hostname is correct")
    print("   2. ✓ Entra ID authentication is enabled on Redis")
    print("   3. ✓ Identity has proper Redis permissions (Data Contributor/Reader)")
    print("   4. ✓ Network connectivity to Azure (check NSGs, firewalls)")
    print("   5. ✓ Azure CLI authenticated: az account show")
    print("   6. ✓ Managed Identity enabled (for Azure services)")
    print("   7. ✓ Service Principal has Redis permissions")
    print("   8. ✓ Key Vault permissions for secret access")
    print("\n🆔 Test identity:")
    print("   az account get-access-token --resource=https://redis.azure.com/")

## 6. Advanced AIMemory with Channel Management

Let's explore enterprise-grade memory management with channels:

In [ ]:
# Create enterprise channels configuration
def create_enterprise_channels_config():
    """Create enterprise-grade channels configuration."""
    channels_content = """
channels:
  working:
    ttl_sec: 14400        # 4 hours - working memory
    encrypt: true         # Encrypt sensitive data
    quota_bytes: 100000   # 100KB limit

  longterm:
    ttl_sec: 604800       # 7 days - important information retention
    encrypt: true         # Always encrypt long-term data
    quota_bytes: 500000   # 500KB limit

  compliance:
    ttl_sec: 2592000      # 30 days - compliance and audit data
    encrypt: true         # Required for compliance
    quota_bytes: 1000000  # 1MB limit

  routing:
    ttl_sec: 86400        # 24 hours - conversation routing
    encrypt: false        # Performance optimization for routing data
    quota_bytes: 50000    # 50KB limit

  debug:
    ttl_sec: 3600         # 1 hour - debugging information
    encrypt: false        # Debug data, shorter retention
    quota_bytes: 25000    # 25KB limit

  security:
    ttl_sec: 86400        # 24 hours - security events and logs
    encrypt: true         # Always encrypt security data
    quota_bytes: 100000   # 100KB limit
"""
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        f.write(channels_content.strip())
        return f.name

channels_file = create_enterprise_channels_config()
print(f"📋 Created enterprise channels configuration: {channels_file}")
print("\n🏢 Enterprise channels include:")
print("   • working: Short-term conversation context")
print("   • longterm: Important persistent information")
print("   • compliance: Audit and compliance data (30-day retention)")
print("   • routing: Conversation flow and agent routing")
print("   • debug: Development and troubleshooting data")
print("   • security: Security events and access logs")

In [ ]:
# Initialize enterprise AIMemory with Azure Redis and channels
try:
    # Get connection details from current config
    hostname = config_to_use['redis_config']['azure_hostname']
    
    # Create AIMemory with enterprise configuration
    if 'azure_keyvault_url' in config_to_use['redis_config']:
        # Key Vault configuration
        memory = AIMemory(
            channels_yaml=channels_file,
            azure_hostname=hostname,
            azure_keyvault_url=config_to_use['redis_config']['azure_keyvault_url'],
            azure_secret_name=config_to_use['redis_config']['azure_secret_name']
        )
        auth_method = "Key Vault"
    elif config_to_use['redis_config'].get('azure_client_secret'):
        # Service Principal configuration
        memory = AIMemory(
            channels_yaml=channels_file,
            azure_hostname=hostname,
            use_azure_entra_id=True,
            azure_client_id=config_to_use['redis_config']['azure_client_id'],
            azure_client_secret=config_to_use['redis_config']['azure_client_secret'],
            azure_tenant_id=config_to_use['redis_config']['azure_tenant_id'],
            azure_username=config_to_use['redis_config']['azure_username']
        )
        auth_method = "Service Principal"
    else:
        # Managed Identity configuration
        memory = AIMemory(
            channels_yaml=channels_file,
            azure_hostname=hostname,
            use_azure_entra_id=True,
            azure_username=config_to_use['redis_config']['azure_username']
        )
        auth_method = "Managed Identity"
    
    print(f"🧠 Enterprise AIMemory initialized with {auth_method}!")
    
    # List available channels
    channels = memory.list_channels()
    print("\n📺 Enterprise memory channels:")
    for name, config in channels.items():
        ttl_hours = config.ttl_sec // 3600
        ttl_days = config.ttl_sec // 86400
        
        if ttl_days >= 1:
            ttl_display = f"{ttl_days}d"
        else:
            ttl_display = f"{ttl_hours}h"
            
        quota_kb = config.quota_bytes // 1000 if config.quota_bytes else "unlimited"
        encryption = "🔒" if config.encrypt else "📄"
        print(f"   {encryption} {name:12} | {ttl_display:4} TTL | {quota_kb:4}KB quota")
    
except Exception as e:
    print(f"❌ Enterprise AIMemory initialization failed: {e}")
    print("   Make sure to replace hostname placeholders with actual values")
    print("   Verify Entra ID authentication is working")

## 7. Enterprise Conversation Management

Let's demonstrate enterprise-grade conversation handling:

In [ ]:
# Create enterprise memory scope
enterprise_scope = MemoryScope(
    agent_id="compliance_advisor",
    user_id="enterprise_client_001",
    session_id="compliance_review_2024_q2",
    env="production"
)

print(f"🎯 Enterprise memory scope created:")
print(f"   Agent: {enterprise_scope.agent_id}")
print(f"   User: {enterprise_scope.user_id}")
print(f"   Session: {enterprise_scope.session_id}")
print(f"   Environment: {enterprise_scope.env}")
print(f"\n🔐 Authentication: {auth_method}")
print(f"🏢 Security: Enterprise-grade with encryption")

In [ ]:
# Write enterprise data to different channels
print("✍️  Writing enterprise data to memory channels...\n")

try:
    # Working memory - current conversation context
    working_id = memory.write(
        scope=enterprise_scope,
        channel="working",
        content="Client requesting compliance review for data processing activities. "
                "Current focus: GDPR compliance for EU customer data. "
                "Client has 50K+ EU customers with personal data processing."
    )
    print(f"💼 Working memory: {working_id[:8]}... (encrypted)")
    
    # Long-term memory - important client information
    longterm_id = memory.write(
        scope=enterprise_scope,
        channel="longterm",
        content="Enterprise client profile: Fortune 500 company, Financial Services sector. "
                "Compliance requirements: SOC2 Type II, PCI DSS, GDPR, CCPA. "
                "Data residency: US, EU. Annual compliance audit: Q4 2024."
    )
    print(f"🏦 Long-term memory: {longterm_id[:8]}... (encrypted)")
    
    # Compliance data - audit trail and regulatory information
    compliance_id = memory.write(
        scope=enterprise_scope,
        channel="compliance",
        content=json.dumps({
            "audit_id": "AUDIT-2024-Q2-001",
            "compliance_frameworks": ["GDPR", "CCPA", "SOC2", "PCI-DSS"],
            "data_classification": "Confidential",
            "retention_policy": "7_years",
            "last_review": "2024-01-15",
            "next_review": "2024-07-15",
            "risk_level": "Medium"
        })
    )
    print(f"📋 Compliance data: {compliance_id[:8]}... (encrypted, 30-day retention)")
    
    # Security event logging
    security_id = memory.write(
        scope=enterprise_scope,
        channel="security",
        content=json.dumps({
            "event_type": "data_access",
            "timestamp": "2024-07-11T10:30:00Z",
            "user_id": "enterprise_client_001",
            "agent_id": "compliance_advisor",
            "action": "compliance_review_initiated",
            "ip_address": "203.0.113.1",
            "user_agent": "NexaMem-Enterprise/1.0",
            "session_id": "compliance_review_2024_q2"
        })
    )
    print(f"🛡️  Security log: {security_id[:8]}... (encrypted)")
    
    # Routing information - for conversation flow
    routing_id = memory.write(
        scope=enterprise_scope,
        channel="routing",
        content=json.dumps({
            "topic": "compliance_review",
            "confidence": 0.98,
            "next_agent": "gdpr_specialist",
            "escalation_level": "standard",
            "priority": "high",
            "client_tier": "enterprise"
        })
    )
    print(f"🗺️  Routing info: {routing_id[:8]}... (performance optimized)")
    
    print("\n✅ All enterprise data written successfully!")
    print("🔒 Sensitive data encrypted and stored securely in Azure Redis")
    
except Exception as e:
    print(f"❌ Enterprise memory write error: {e}")

## 8. Enterprise Analytics and Monitoring

Monitor memory usage and security across enterprise channels:

In [ ]:
# Enterprise memory usage analytics
print("📊 Enterprise Memory Analytics\n")

try:
    channels = memory.list_channels()
    
    total_bytes = 0
    encrypted_bytes = 0
    compliance_data = []
    
    print("📈 Channel Usage Report:")
    print("   Channel      | Msgs | Bytes  |  KB  | Encryption | TTL  | Quota")
    print("   " + "-" * 65)
    
    for channel_name, channel_config in channels.items():
        try:
            byte_count = memory.count_bytes(enterprise_scope, channel_name)
            total_bytes += byte_count
            
            if channel_config.encrypt:
                encrypted_bytes += byte_count
            
            # Get channel messages for count
            messages, meta = memory.read(enterprise_scope, channel_name, max_msgs=1000)
            msg_count = meta['total_messages']
            
            # Format display values
            kb_count = byte_count / 1024
            encryption_status = "🔒 Yes" if channel_config.encrypt else "📄 No"
            ttl_hours = channel_config.ttl_sec // 3600
            ttl_days = channel_config.ttl_sec // 86400
            
            if ttl_days >= 1:
                ttl_display = f"{ttl_days:2}d"
            else:
                ttl_display = f"{ttl_hours:2}h"
            
            quota_kb = (channel_config.quota_bytes // 1000) if channel_config.quota_bytes else 0
            quota_display = f"{quota_kb}KB" if quota_kb > 0 else "unlimited"
            
            print(f"   {channel_name:12} | {msg_count:4} | {byte_count:6} | {kb_count:4.1f} | {encryption_status:7} | {ttl_display} | {quota_display}")
            
            # Collect compliance data
            if channel_name in ['compliance', 'security', 'longterm']:
                compliance_data.append({
                    'channel': channel_name,
                    'messages': msg_count,
                    'bytes': byte_count,
                    'encrypted': channel_config.encrypt,
                    'ttl_days': ttl_days
                })
            
        except Exception as e:
            print(f"   {channel_name:12} | Error: {e}")
    
    print("   " + "-" * 65)
    print(f"\n📊 Enterprise Summary:")
    print(f"   Total memory usage: {total_bytes:,} bytes ({total_bytes/1024:.1f} KB)")
    print(f"   Encrypted data: {encrypted_bytes:,} bytes ({(encrypted_bytes/total_bytes*100) if total_bytes > 0 else 0:.1f}%)")
    print(f"   Authentication: {auth_method}")
    print(f"   Compliance channels: {len(compliance_data)} active")
    
    # Compliance and security summary
    if compliance_data:
        print(f"\n🛡️  Compliance & Security:")
        for data in compliance_data:
            encryption_icon = "🔒" if data['encrypted'] else "⚠️"
            print(f"   {encryption_icon} {data['channel']:12}: {data['messages']:3} msgs, {data['ttl_days']:2}d retention")
    
except Exception as e:
    print(f"❌ Analytics error: {e}")

## 9. Connection Health and Security Validation

Validate enterprise connection security and health:

In [ ]:
# Enterprise connection health and security check
try:
    from nexamem.config import get_config
    
    config = get_config()
    if hasattr(config, 'redis_adapter') and config.redis_adapter:
        adapter = config.redis_adapter
        
        # Connection health
        ping_result = adapter.ping()
        print(f"🏓 Redis connection: {'✅ Success' if ping_result else '❌ Failed'}")
        
        # Connection security information
        info = adapter.get_info()
        print(f"\n🔗 Enterprise Connection Details:")
        print(f"   Host: {info.get('host', 'unknown')}")
        print(f"   Port: {info.get('port', 'unknown')} (SSL)")
        print(f"   SSL Enabled: {info.get('ssl', 'unknown')}")
        print(f"   Database: {info.get('db', 'unknown')}")
        print(f"   Azure Redis: {info.get('is_azure', 'unknown')}")
        print(f"   Connected: {info.get('connected', 'unknown')}")
        
        # Security and compliance features
        print(f"\n🛡️  Security & Compliance:")
        print(f"   🔐 Authentication: {auth_method}")
        print(f"   🔒 Encryption: In-transit (SSL) + At-rest (channel-based)")
        print(f"   🏢 Identity Provider: Microsoft Entra ID")
        print(f"   📋 Audit Trail: Security and compliance channels")
        print(f"   🗂️  Data Classification: Confidential/Enterprise")
        print(f"   ⏰ Retention Policies: Channel-specific TTL (1h to 30d)")
        print(f"   📊 Quota Management: Per-channel byte limits")
        
        # Enterprise recommendations
        print(f"\n✅ Enterprise Recommendations:")
        print(f"   • Monitor Azure Redis metrics in Azure Portal")
        print(f"   • Set up alerts for connection failures and memory usage")
        print(f"   • Review compliance data regularly (30-day retention)")
        print(f"   • Rotate Service Principal secrets periodically")
        print(f"   • Enable Azure Redis logs for security monitoring")
        
    else:
        print("❌ No Redis adapter found in configuration")
        
except Exception as e:
    print(f"❌ Enterprise health check error: {e}")

## 10. Enterprise Data Cleanup

Secure cleanup of enterprise demonstration data:

In [ ]:
# Enterprise data cleanup with audit trail
cleanup_confirmed = input("Do you want to clean up enterprise demo data? (y/N): ").lower().strip()

if cleanup_confirmed == 'y':
    try:
        print("🧹 Starting enterprise data cleanup...")
        
        # Log cleanup action to security channel first
        cleanup_log_id = memory.write(
            scope=enterprise_scope,
            channel="security",
            content=json.dumps({
                "event_type": "data_cleanup",
                "timestamp": "2024-07-11T11:00:00Z",
                "user_id": "enterprise_client_001",
                "action": "demo_data_cleanup_initiated",
                "scope": str(enterprise_scope.__dict__),
                "reason": "End of demonstration session"
            })
        )
        print(f"📝 Cleanup logged to security channel: {cleanup_log_id[:8]}...")
        
        # Count data before cleanup
        total_before = 0
        channel_counts = {}
        
        for channel_name in memory.list_channels().keys():
            try:
                byte_count = memory.count_bytes(enterprise_scope, channel_name)
                total_before += byte_count
                channel_counts[channel_name] = byte_count
            except:
                channel_counts[channel_name] = 0
        
        print(f"📊 Data before cleanup: {total_before:,} bytes across {len(channel_counts)} channels")
        
        # Perform bulk erase
        deleted_count = memory.bulk_erase(enterprise_scope)
        print(f"🗑️  Deleted {deleted_count} keys from Redis")
        
        # Verify cleanup
        total_after = 0
        for channel_name in memory.list_channels().keys():
            try:
                byte_count = memory.count_bytes(enterprise_scope, channel_name)
                total_after += byte_count
            except:
                pass
        
        print(f"📊 Data after cleanup: {total_after:,} bytes")
        print(f"♻️  Cleanup efficiency: {((total_before - total_after) / total_before * 100) if total_before > 0 else 100:.1f}%")
        print("✅ Enterprise demo data cleanup completed securely!")
        
        # Security note
        print("\n🔒 Security Note:")
        print("   • Cleanup action logged in security channel")
        print("   • Encrypted data securely removed from Azure Redis")
        print("   • Audit trail maintained for compliance")
        
    except Exception as e:
        print(f"❌ Enterprise cleanup error: {e}")
        print("   Data will naturally expire based on TTL settings")
        print("   Security and compliance data retained per policy")
else:
    print("ℹ️  Skipping cleanup - enterprise demo data will remain")
    print("   💡 Data will automatically expire based on enterprise TTL policies:")
    print("      • Debug: 1 hour")
    print("      • Routing: 24 hours")
    print("      • Working: 4 hours")
    print("      • Security: 24 hours")
    print("      • Long-term: 7 days")
    print("      • Compliance: 30 days")

# Clean up temporary files
try:
    os.unlink(channels_file)
    print("🗑️  Temporary channels configuration file removed")
except:
    pass

## 🎯 Enterprise Summary

Congratulations! You've successfully implemented enterprise-grade NexaMem with Azure Redis:

### ✅ **Enterprise Features Demonstrated:**

#### 🔐 **Security & Authentication:**
- **Entra ID Integration**: Identity-based authentication with Azure AD
- **Multiple Auth Methods**: Managed Identity, Service Principal, Key Vault
- **Zero Secrets**: Managed Identity eliminates hardcoded credentials
- **Token Rotation**: Automatic token refresh and rotation

#### 🏢 **Enterprise Memory Management:**
- **Channel-based Organization**: Separate channels for different data types
- **Encryption at Rest**: Configurable per-channel encryption
- **Retention Policies**: Customizable TTL from 1 hour to 30 days
- **Quota Management**: Per-channel byte limits and monitoring

#### 📋 **Compliance & Governance:**
- **Audit Trails**: Comprehensive logging in security channels
- **Data Classification**: Confidential/Enterprise data handling
- **Compliance Channels**: Dedicated 30-day retention for regulatory data
- **Secure Cleanup**: Audited data removal procedures

#### 📊 **Monitoring & Analytics:**
- **Usage Analytics**: Memory consumption and performance monitoring
- **Security Monitoring**: Connection health and security validation
- **Compliance Reporting**: Channel-specific compliance metrics

### 🚀 **Production Deployment Checklist:**

#### ✅ **Security Configuration:**
- [ ] Configure Managed Identity on Azure services
- [ ] Set up Service Principal with minimal required permissions
- [ ] Enable Azure Redis Entra ID authentication
- [ ] Configure Key Vault for secret management (if needed)
- [ ] Set up network security groups and firewall rules

#### ✅ **Monitoring & Alerting:**
- [ ] Enable Azure Redis diagnostics and logging
- [ ] Set up Azure Monitor alerts for connection failures
- [ ] Configure memory usage alerts and quotas
- [ ] Implement application-level health checks
- [ ] Set up security event monitoring

#### ✅ **Compliance & Governance:**
- [ ] Define data retention policies per regulation
- [ ] Configure encryption for sensitive channels
- [ ] Set up audit log collection and retention
- [ ] Document data classification and handling procedures
- [ ] Implement regular compliance reviews

#### ✅ **Operational Excellence:**
- [ ] Implement infrastructure as code (ARM/Bicep/Terraform)
- [ ] Set up CI/CD pipelines with proper secret management
- [ ] Configure backup and disaster recovery procedures
- [ ] Plan for credential rotation and updates
- [ ] Establish incident response procedures

### 📚 **Enterprise Resources:**

- [Azure Cache for Redis Enterprise Documentation](https://docs.microsoft.com/en-us/azure/azure-cache-for-redis/)
- [Microsoft Entra ID Authentication](https://docs.microsoft.com/en-us/azure/active-directory/)
- [Azure Key Vault Best Practices](https://docs.microsoft.com/en-us/azure/key-vault/)
- [Azure Security Center](https://docs.microsoft.com/en-us/azure/security-center/)
- [Azure Compliance Documentation](https://docs.microsoft.com/en-us/azure/compliance/)

### 🔄 **Next Steps:**

1. **Customize** channel configurations for your specific use case
2. **Integrate** with your existing identity and security infrastructure
3. **Deploy** to production with proper security and monitoring
4. **Monitor** performance and security metrics continuously
5. **Review** compliance and audit requirements regularly

---

**🏢 Ready for enterprise deployment with NexaMem and Azure Redis! 🚀**